In [1]:
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML



# ======================================================
# ヘルパー関数
# ======================================================
def initialize_opinion_grid(n):
    """
    n x n のグリッドを用意し、各セルに [0,1) の一様乱数をセット。
    これを「意見 (opinion)」とする。
    """
    return np.random.rand(n, n)

def get_random_neighbor4(n, x, y):
    """
    上下左右4近傍からランダムに1つ返す。
    トーラス状（境界はつながっている）とする。
    """
    neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    dx, dy = random.choice(neighbors)
    nx = (x + dx) % n
    ny = (y + dy) % n
    return nx, ny

def step_opinion_one_pair(grid):
    """
    グリッド中のランダムに選ばれた1セルと、その4近傍（またはランダム）の
    1セルのみを更新する。
    """
    n = grid.shape[0]
    
    # ランダムに1セルを選ぶ
    x = random.randint(0, n - 1)
    y = random.randint(0, n - 1)
    
    # 相手セルを選ぶ
    if spatial_interactions:
        nx, ny = get_random_neighbor4(n, x, y)
    else:
        nx = random.randint(0, n - 1)
        ny = random.randint(0, n - 1)

    # old状態を取得
    x1 = grid[x, y]
    x2 = grid[nx, ny]
    diff = abs(x1 - x2)

    # --- 正の影響 (意見差が閾値以内) ---
    if diff < confidence_threshold:
        x1_new = x1 + learning_rate * (x2 - x1)
        x2_new = x2 + learning_rate * (x1 - x2)
        grid[x, y]   = x1_new
        grid[nx, ny] = x2_new

    # --- 負の影響 (反目) ---
    elif repulsion and diff > confidence_threshold:
        if x1 > x2:
            # x1 が大きい
            x1_new = x1 + learning_rate * (x1 - x2) * (1 - x1) * 0.5
            x2_new = x2 + learning_rate * (x2 - x1) * (1 + x2) * 0.5
        else:
            # x2 が大きい
            x1_new = x1 + learning_rate * (x1 - x2) * (1 + x1) * 0.5
            x2_new = x2 + learning_rate * (x2 - x1) * (1 - x2) * 0.5
        
        grid[x, y]   = x1_new
        grid[nx, ny] = x2_new
    # それ以外の場合は何もしない (diff == confidence_threshold とか、repulsion=Falseの場合など)

def run_simulation_one_pair(num_steps=num_steps, store_interval=store_interval):
    """
    各ステップで「1ペア」だけを更新し、
    store_interval (例:100) ステップごとに状態を保存する。
    """
    # 初期グリッド作成
    grid = initialize_opinion_grid(N)
    
    # 0ステップ目（初期状態）を格納
    grid_states = [grid.copy()]
    time_points = [0]
    
    # シミュレーション実行
    for t in range(1, num_steps+1):
        step_opinion_one_pair(grid)  # 1ペアだけ更新
        # 100ステップごとに格納
        if t % store_interval == 0:
            grid_states.append(grid.copy())
            time_points.append(t)
    
    return grid_states, time_points

def animate_snapshots(grid_states, time_points):
    """
    2つのサブプロットでアニメーション表示:
      - 左: グリッドの意見分布 (imshow) (各ステップをスナップショット表示)
      - 右: 各ステップ(スナップショット)における全セル意見値 (scatter)
            (この例では "累積" ではなく「そのステップのみ」を描画)
    """
    # grid_states: [state0, state1, state2, ...]
    # time_points: [0, 100, 200, ...] など
    num_frames = len(grid_states)
    
    fig = plt.figure(figsize=(12,5))
    ax1 = fig.add_subplot(1,2,1)
    ax2 = fig.add_subplot(1,2,2)
    
    # 左: imshow
    im = ax1.imshow(grid_states[0], vmin=0, vmax=1, cmap=plt.cm.coolwarm)
    ax1.set_title("Grid of Opinions")
    ax1.set_xticks([])
    ax1.set_yticks([])
    
    # 右: 散布図
    ax2.set_title("All Agents' Opinions (Snapshots)")
    ax2.set_xlim(0, time_points[-1])  # 最後の時刻まで
    ax2.set_ylim(0, 1)
    ax2.set_xlabel("Time Step")
    ax2.set_ylabel("Opinion Value")
    
    # 初期描画（frame=0のスナップショット）
    x_vals_init = np.full(N*N, time_points[0])   # すべて time_points[0]
    y_vals_init = grid_states[0].flatten()
    scatter = ax2.scatter(x_vals_init, y_vals_init, s=5, marker='o')
    
    def update(frame_index):
        """
        frame_index は 0,1,2,...,num_frames-1
        """
        # グリッド更新
        im.set_data(grid_states[frame_index])
        
        # 散布図更新（このスナップショット分のみプロット）
        t = time_points[frame_index]
        opinions_flat = grid_states[frame_index].flatten()
        x_vals = np.full(opinions_flat.shape, t)
        y_vals = opinions_flat
        scatter.set_offsets(np.c_[x_vals, y_vals])
        
        return [im, scatter]
    
    ani = animation.FuncAnimation(
        fig,
        update,
        frames=num_frames,       # 保存したスナップショットの数だけアニメーション
        interval=100,           # アニメーション切り替え間隔(ミリ秒)
        blit=False
    )
    return HTML(ani.to_jshtml())



NameError: name 'num_steps' is not defined

In [2]:
# ======================================================
# グローバルパラメータ例
# ======================================================
N = 21                # グリッドの大きさ (N x N)
learning_rate = 0.3    # 学習率
confidence_threshold = 0.2  # 意見の差がこれ以下なら正の影響
spatial_interactions = True  # True なら4近傍から相手を選ぶ
repulsion = True       # True なら「反目（意見がさらに遠ざかる）」を考慮
num_steps = 30000
store_interval = 100

# ======================================================
# 実行例
# ======================================================
if __name__ == "__main__":
    # シミュレーションを 2000 ステップ実行し、100ステップごとに保存
    grid_states, time_points = run_simulation_one_pair(num_steps=num_steps, store_interval=store_interval)
    
    # アニメーション表示
    #anim = animate_snapshots(grid_states, time_points)
    #display(anim)


NameError: name 'run_simulation_one_pair' is not defined

In [3]:
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML



# ================================
# ヘルパー関数
# ================================
def initialize_opinion_grid(n):
    return np.random.rand(n, n)

def get_random_neighbor4(n, x, y):
    neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    dx, dy = random.choice(neighbors)
    nx = (x + dx) % n
    ny = (y + dy) % n
    return nx, ny

def step_opinion_one_pair(grid):
    n = grid.shape[0]
    x = random.randint(0, n - 1)
    y = random.randint(0, n - 1)
    
    if spatial_interactions:
        nx, ny = get_random_neighbor4(n, x, y)
    else:
        nx = random.randint(0, n - 1)
        ny = random.randint(0, n - 1)
    
    x1 = grid[x, y]
    x2 = grid[nx, ny]
    diff = abs(x1 - x2)
    
    # 正の影響
    if diff < confidence_threshold:
        x1_new = x1 + learning_rate * (x2 - x1)
        x2_new = x2 + learning_rate * (x1 - x2)
        grid[x, y]   = x1_new
        grid[nx, ny] = x2_new
    
    # 負の影響
    elif repulsion and diff > confidence_threshold:
        if x1 > x2:
            x1_new = x1 + learning_rate * (x1 - x2) * (1 - x1) * 0.5
            x2_new = x2 + learning_rate * (x2 - x1) * (1 + x2) * 0.5
        else:
            x1_new = x1 + learning_rate * (x1 - x2) * (1 + x1) * 0.5
            x2_new = x2 + learning_rate * (x2 - x1) * (1 - x2) * 0.5
        grid[x, y]   = x1_new
        grid[nx, ny] = x2_new

def run_simulation_one_pair(num_steps=num_steps, store_interval=store_interval):
    """
    各ステップで 1ペアだけを更新し、store_interval ステップごとに状態を保存。
    """
    grid = initialize_opinion_grid(N)
    grid_states = [grid.copy()]
    time_points = [0]
    
    for t in range(1, num_steps+1):
        step_opinion_one_pair(grid)
        if t % store_interval == 0:
            grid_states.append(grid.copy())
            time_points.append(t)
    
    return grid_states, time_points

def animate_snapshots(grid_states, time_points):
    """
    左側: imshow で各スナップショット
    右側: 全スナップショット(0～frame)の意見値を累積表示
    """
    num_frames = len(grid_states)  # スナップショットの個数
    
    # --- 事前に「全フレーム分の x,y を結合」しておき、フレームi に対して
    #     0～i のスナップショット相当を使えるようにする ---

    # フレームごとの (x, y) データをリストにためる
    all_x_list = []
    all_y_list = []
    for i, grid in enumerate(grid_states):
        # grid は shape (N, N)
        opinions_flat = grid.flatten()
        # そのスナップショットの「時刻(time_points[i])」を x とする
        x_vals = np.full(opinions_flat.shape, time_points[i])
        y_vals = opinions_flat
        all_x_list.append(x_vals)
        all_y_list.append(y_vals)
    
    # numpy配列にまとめる
    all_x = np.concatenate(all_x_list)  # shape: (num_frames * N*N, )
    all_y = np.concatenate(all_y_list)  # 同上
    
    # フレーム i に対応する累積インデックスを計算
    # 各フレームで、(フレーム0 + ... + フレームi) までの点数は (i+1)* (N*N)
    points_per_frame = N * N
    cumulative_indices = []
    for i in range(num_frames):
        end_idx = (i+1) * points_per_frame
        cumulative_indices.append(end_idx)

    # --- Figure, Axes ---
    fig = plt.figure(figsize=(12,5))
    ax1 = fig.add_subplot(1,2,1)
    ax2 = fig.add_subplot(1,2,2)
    
    # 左 (imshow)
    im = ax1.imshow(grid_states[0], vmin=0, vmax=1, cmap=plt.cm.coolwarm)
    ax1.set_title("Grid of Opinions")
    ax1.set_xticks([])
    ax1.set_yticks([])
    
    # 右 (scatter; 累積表示)
    ax2.set_title("All Agents' Opinions (Cumulative Snapshots)")
    ax2.set_xlim(0, time_points[-1])
    ax2.set_ylim(0, 1)
    ax2.set_xlabel("Time Step")
    ax2.set_ylabel("Opinion Value")
    
    # 初期描画 (frame=0)
    # => フレーム0のみのデータ点 => cumulative_indices[0] = 1*N*N
    end0 = cumulative_indices[0]  # = N*N
    scatter = ax2.scatter(all_x[:end0], all_y[:end0], s=5, marker='o')
    
    def update(frame):
        """
        frame は 0,1,2,...,num_frames-1
        """
        # 左 subplot: imshowのアップデート
        im.set_data(grid_states[frame])
        
        # 右 subplot: 0～frame のデータ点までを表示 (累積)
        end_index = cumulative_indices[frame]
        scatter.set_offsets(np.c_[all_x[:end_index], all_y[:end_index]])
        
        return [im, scatter]
    
    ani = animation.FuncAnimation(
        fig,
        update,
        frames=num_frames,
        interval=400,  # ms
        blit=False
    )
    return HTML(ani.to_jshtml())




In [4]:
# ================================
# グローバルパラメータ (例)
# ================================
N = 21                 # グリッドの大きさ
learning_rate = 0.3    # 学習率
confidence_threshold = 0.1
spatial_interactions = True
repulsion = True
num_steps = 20000
store_interval = 100
# -----------------------------
# 実行例
# -----------------------------
if __name__ == "__main__":
    grid_states, time_points = run_simulation_one_pair(num_steps=num_steps, store_interval=store_interval)
    #anim = animate_snapshots(grid_states, time_points)
    #display(anim)